# Machine Unlearning on Yelp2018: All 4 Pipelines

This notebook runs **all four** unlearning pipelines sequentially on the same pretrained LightGCN checkpoint.

**Order (fastest → slowest):**
1. **AIE** – Attention-Based Influence Encoder (~2.5M params, 3 losses)
2. **CIE** – Causal Influence Encoder (~2.5M params, 5 losses)
3. **GAIE** – Graph Autoencoder Influence Encoder (VAE, 4 losses)
4. **HIE** – Hypernetwork-Based Influence Encoder (~40M params, 3 losses)

Each pipeline: **Unlearn → Fine-tune → Evaluate (Recall, NDCG, MI-BF, MI-NG)**

### Kaggle Setup
Enable **GPU accelerator** (Settings → Accelerator → GPU).

### Experimental protocol (aligned with UnlearnRec, SIGIR'25, Sec. 4.1.4)

**Threat model / unlearning target.** Adversarial edges are the least-probable user-item pairs
under a GCN trained on the clean data. The backbone LightGCN is trained **on the attacked graph**
(clean edges + injected adversarial edges), so the adversarial edges are genuinely learned as
positives. The unlearning task is to remove exactly those edges from the trained backbone.

**Why this matters.** If the backbone were trained on the clean graph instead, the adversarial
edges would never have been learned, the "before unlearning" scores would already be at or below
negative-sample level, and MI-BF / MI-NG would not measure forgetting. This notebook therefore
pretrains with `adversarial_attack=True`.

**Ground truth.** The exact-unlearning reference ("Retrain") is the same architecture retrained
from scratch on the residual graph (attacked graph minus adversarial edges = the clean data).

**Metrics.** MI-BF = mean recommendation probability of the unlearned edges before vs. after
unlearning (higher is better, must be > 1). MI-NG = mean probability of negative samples vs.
unlearned edges after unlearning (> 1 means unlearned edges are now less recommendable than
random non-edges). We also log the raw before/after/negative probabilities as a sanity check
that MI-NG is not trivially pre-satisfied.


---
## 0. Environment Setup

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed ✓")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from Utils.utils import innerProduct, cal_mi_metrics, print_args
from models.Model import LightGCN, AIE, CIE, GAIE, HIE

print("All imports successful!")

In [ ]:
import torch as t
import numpy as np
import random
import time

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")
    print(f"Memory: {t.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 1. Pretrain LightGCN on Yelp2018

Shared across all 4 pipelines. Only needs to run **once**.

In [ ]:
# ============================================================
# Hyperparameters for LightGCN pretraining on Yelp2018
# ============================================================

args.data = 'yelp2018'
args.model = 'lightgcn'
args.gpu = '0'
args.seed = 1234
args.lr = 1e-3
args.batch = 4096
args.epoch = 350
args.latdim = 128
args.gnn_layer = 3
args.reg = 1e-7
args.topk = 20
args.tst_epoch = 3
args.tst_bat = 256
args.decay = 1.0
args.bpr_wei = 1.0
# PAPER PROTOCOL (UnlearnRec Sec 4.1.4): the backbone to be unlearned must be
# trained on the ATTACKED graph so the adversarial edges are genuinely learned.
args.adversarial_attack = True
args.adv_method = 'lightgcn'
args.save_path = './ckpt/pretrain_yelp2018_adv'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu

print_args(args)

In [ ]:
handler = DataHandler()
handler.load_data(drop_rate=0.0, adv_attack=True)
# dropped_edges == the injected adversarial edges (the unlearning target)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges: {handler.trn_loader.dataset.__len__()}")
print(f"Test users: {handler.tst_loader.dataset.__len__()}")
print(f"Injected adversarial edges: {len(handler.adv_edges[0])}")

In [ ]:
from training.pretrain_lightgcn import Coach as PretrainCoach

pretrain_coach = PretrainCoach(handler)
pretrain_coach.run()

print("\n" + "="*60)
print("Pretraining complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

In [ ]:

# Load the best checkpoint (saved when Recall improved during training)
PRETRAINED_PATH = args.save_path
ckp = t.load(PRETRAINED_PATH + '.mod', weights_only=False)
pretrained_model = ckp['model'].cuda()
pretrained_model.eval()

reses = pretrain_coach.tst_epoch(pretrained_model)
print(f"\nBest Pretrained LightGCN Performance:")
print(f"  Recall@{args.topk}: {reses['Recall']:.4f}")
print(f"  NDCG@{args.topk}:   {reses['NDCG']:.4f}")


---
# Results Collection

We'll store results from each pipeline for a final comparison.

In [ ]:
# Dictionary to collect results from all pipelines
all_results = {}

---
## 1b. Retrain reference (exact-unlearning ground truth)

The paper's "Retrain" row: the same backbone retrained from scratch on the residual graph
(= clean training data). Used as ground truth for both utility (Recall/NDCG) and efficacy
(MI-BF/MI-NG), and as the efficiency yardstick (unlearning must beat this wall-clock time).

In [ ]:
# ============================================================
# Retrain reference (exact unlearning): train from scratch on E_r
# ============================================================
RUN_RETRAIN_REFERENCE = True   # set False to skip and save GPU time

if RUN_RETRAIN_REFERENCE:
    args.adversarial_attack = False
    args.save_path = './ckpt/retrain_ref'
    handler_clean = DataHandler()
    handler_clean.load_data(drop_rate=0.0, adv_attack=False)

    retrain_coach = PretrainCoach(handler_clean)
    t.cuda.reset_peak_memory_stats()
    _t0 = time.time()
    retrain_coach.run()
    retrain_time = time.time() - _t0
    retrain_mem = t.cuda.max_memory_allocated() / 1e9

    ckp_r = t.load('./ckpt/retrain_ref.mod', weights_only=False)
    retrain_model = ckp_r['model'].cuda()
    retrain_model.eval()
    retrain_metrics = retrain_coach.tst_epoch(retrain_model)
    print(f"\n[Retrain] Recall@{args.topk}: {retrain_metrics['Recall']:.4f}, "
          f"NDCG@{args.topk}: {retrain_metrics['NDCG']:.4f}, "
          f"time: {retrain_time:.1f}s, peak mem: {retrain_mem:.2f} GB")


In [ ]:
if RUN_RETRAIN_REFERENCE:
    # MI metrics of the retrained model on the adversarial edges.
    # "Before" scores come from the attacked backbone (the model being unlearned).
    args.adversarial_attack = True
    handler_adv_eval = DataHandler()
    handler_adv_eval.load_data(drop_rate=0.0, adv_attack=True)
    adv_u, adv_i = handler_adv_eval.dropped_edges

    with t.no_grad():
        ret_u_emb, ret_i_emb = retrain_model.forward(handler_clean.ts_ori_adj, keepRate=1.0)
        atk_u_emb, atk_i_emb = pretrained_model.forward(handler_adv_eval.ts_ori_adj, keepRate=1.0)

    ret_drp = innerProduct(ret_u_emb[adv_u], ret_i_emb[adv_i])
    atk_drp = innerProduct(atk_u_emb[adv_u], atk_i_emb[adv_i])

    rows, cols = handler_adv_eval.ori_trn_mat.row, handler_adv_eval.ori_trn_mat.col
    edge_set = set(zip(rows.tolist(), cols.tolist()))
    neg_r, neg_c = [], []
    while len(neg_r) < len(adv_u):
        i, j = np.random.randint(args.user), np.random.randint(args.item)
        if (i, j) not in edge_set:
            edge_set.add((i, j)); neg_r.append(i); neg_c.append(j)
    ret_neg = innerProduct(ret_u_emb[neg_r], ret_i_emb[neg_c])

    retrain_mi = cal_mi_metrics(ret_drp, ret_neg, before_drp_scores=atk_drp)
    all_results['Retrain'] = {
        'Recall': retrain_metrics['Recall'], 'NDCG': retrain_metrics['NDCG'],
        'MI_BF': retrain_mi['mi_bf'], 'MI_NG': retrain_mi['mi_ng'],
        'BeforeProb': retrain_mi['avg_before_prob'], 'AfterProb': retrain_mi['avg_after_prob'],
        'NegProb': retrain_mi['avg_neg_prob'],
        'UnlearnTime': retrain_time, 'FinetuneTime': 0.0, 'PeakMemGB': retrain_mem,
    }
    print(f"[Retrain] MI-BF={retrain_mi['mi_bf']:.4f}, MI-NG={retrain_mi['mi_ng']:.4f}")

    del retrain_coach, handler_clean
    t.cuda.empty_cache()


---
---
# Pipeline 1: AIE (Attention-Based Influence Encoder)

**Fastest pipeline** — ~2.5M params, 3 losses (BPR + unlearn + alignment).

Deleted edges → GAT over influence graph → MLP shift generator → ΔE

**No reconstruction loss** (unlike GAIE).

## AIE — Step 2a: Unlearn

In [ ]:
# ============================================================
# Hyperparameters for AIE unlearning on Yelp2018
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.3
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_aie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

In [ ]:
handler_unlearn_aie = DataHandler()
handler_unlearn_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_aie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_aie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_aie.picked_edges[0])}")

In [ ]:
from unlearning.aie_unlearn import Coach as AIECoach

aie_coach = AIECoach(handler_unlearn_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_coach.run()
aie_unlearn_time = time.time() - _t0
aie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE unlearn] {aie_unlearn_time:.1f}s, peak GPU mem {aie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## AIE — Step 2b: Fine-Tune

In [ ]:
args.model_2_finetune = './ckpt/yelp_aie_unlearn'

args.fineTune = True
args.epoch = 20
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.2
args.align_wei = 0.01
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_aie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)

In [ ]:
handler_ft_aie = DataHandler()
handler_ft_aie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

aie_ft_coach = AIECoach(handler_ft_aie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
aie_ft_coach.run()
aie_ft_time = time.time() - _t0
aie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[AIE finetune] {aie_ft_time:.1f}s, peak GPU mem {aie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("AIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## AIE — Step 3: Evaluate

In [ ]:
handler_eval_aie = DataHandler()
handler_eval_aie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    aie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned AIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

aie_ft_coach.handler = handler_eval_aie

In [ ]:
aie_metrics = aie_ft_coach.tst_epoch(aie_ft_coach.model)
print(f"\n[AIE] Recall@{args.topk}: {aie_metrics['Recall']:.4f}")
print(f"[AIE] NDCG@{args.topk}:   {aie_metrics['NDCG']:.4f}")

In [ ]:
aie_mi = aie_ft_coach.test_unlearn(aie_ft_coach.model, prefix='[AIE] Final Evaluation')

In [ ]:
all_results['AIE'] = {
    'Recall': aie_metrics['Recall'], 'NDCG': aie_metrics['NDCG'],
    'MI_BF': aie_mi['mi_bf'], 'MI_NG': aie_mi['mi_ng'],
    'BeforeProb': aie_mi['avg_before_prob'], 'AfterProb': aie_mi['avg_after_prob'],
    'NegProb': aie_mi['avg_neg_prob'],
    'UnlearnTime': aie_unlearn_time, 'FinetuneTime': aie_ft_time,
    'PeakMemGB': max(aie_unlearn_mem, aie_ft_mem),
}
print("AIE results stored.")

# Free GPU memory
del aie_coach, aie_ft_coach, handler_unlearn_aie, handler_ft_aie, handler_eval_aie
t.cuda.empty_cache()
print("AIE objects freed, GPU cache cleared.")

---
---
# Pipeline 2: CIE (Causal Influence Encoder)

**Second fastest** — ~2.5M params, 5 losses (BPR + unlearn + alignment + contrastive + causal).

Models unlearning as a **causal intervention** do(e_ij = 0).
Computes **counterfactual embeddings** E^cf once, then trains with extra consistency losses.

CIE-specific params: `contrast_wei=0.01`, `causal_wei=0.1`.

## CIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for CIE unlearning on Yelp2018
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.2
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.contrast_wei = 0.01
args.causal_wei = 0.15
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_cie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_unlearn_cie = DataHandler()
handler_unlearn_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_cie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_cie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_cie.picked_edges[0])}")

In [ ]:
from unlearning.cie_unlearn import Coach as CIECoach

cie_coach = CIECoach(handler_unlearn_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_coach.run()
cie_unlearn_time = time.time() - _t0
cie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE unlearn] {cie_unlearn_time:.1f}s, peak GPU mem {cie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## CIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/yelp_cie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.contrast_wei = 0.005
args.causal_wei = 0.15
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_cie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_ft_cie = DataHandler()
handler_ft_cie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

cie_ft_coach = CIECoach(handler_ft_cie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
cie_ft_coach.run()
cie_ft_time = time.time() - _t0
cie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[CIE finetune] {cie_ft_time:.1f}s, peak GPU mem {cie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("CIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## CIE — Step 3: Evaluate

In [ ]:
handler_eval_cie = DataHandler()
handler_eval_cie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    cie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned CIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

cie_ft_coach.handler = handler_eval_cie

In [ ]:
cie_metrics = cie_ft_coach.tst_epoch(cie_ft_coach.model)
print(f"\n[CIE] Recall@{args.topk}: {cie_metrics['Recall']:.4f}")
print(f"[CIE] NDCG@{args.topk}:   {cie_metrics['NDCG']:.4f}")

In [ ]:
cie_mi = cie_ft_coach.test_unlearn(cie_ft_coach.model, prefix='[CIE] Final Evaluation')

In [ ]:
all_results['CIE'] = {
    'Recall': cie_metrics['Recall'], 'NDCG': cie_metrics['NDCG'],
    'MI_BF': cie_mi['mi_bf'], 'MI_NG': cie_mi['mi_ng'],
    'BeforeProb': cie_mi['avg_before_prob'], 'AfterProb': cie_mi['avg_after_prob'],
    'NegProb': cie_mi['avg_neg_prob'],
    'UnlearnTime': cie_unlearn_time, 'FinetuneTime': cie_ft_time,
    'PeakMemGB': max(cie_unlearn_mem, cie_ft_mem),
}
print("CIE results stored.")

del cie_coach, cie_ft_coach, handler_unlearn_cie, handler_ft_cie, handler_eval_cie
t.cuda.empty_cache()
print("CIE objects freed, GPU cache cleared.")

---
---
# Pipeline 3: GAIE (Graph Autoencoder Influence Encoder)

**Third fastest** — VAE-based, 4 losses (BPR + unlearn + alignment + reconstruction).

Influence graph → VAE encoder → latent z → reparameterize → shift MLP → ΔE

GAIE-specific: `rec_wei=0.1` (reconstruction loss).

## GAIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for GAIE unlearning on Yelp2018
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.5
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_gaie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_unlearn_gaie = DataHandler()
handler_unlearn_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_gaie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_gaie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_gaie.picked_edges[0])}")

In [ ]:
from unlearning.gaie_unlearn import Coach as GAIECoach

gaie_coach = GAIECoach(handler_unlearn_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_coach.run()
gaie_unlearn_time = time.time() - _t0
gaie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE unlearn] {gaie_unlearn_time:.1f}s, peak GPU mem {gaie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## GAIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/yelp_gaie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.1
args.rec_wei = 0.03
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_gaie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_ft_gaie = DataHandler()
handler_ft_gaie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

gaie_ft_coach = GAIECoach(handler_ft_gaie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
gaie_ft_coach.run()
gaie_ft_time = time.time() - _t0
gaie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[GAIE finetune] {gaie_ft_time:.1f}s, peak GPU mem {gaie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("GAIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## GAIE — Step 3: Evaluate

In [ ]:
handler_eval_gaie = DataHandler()
handler_eval_gaie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    gaie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned GAIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

gaie_ft_coach.handler = handler_eval_gaie

In [ ]:
gaie_metrics = gaie_ft_coach.tst_epoch(gaie_ft_coach.model)
print(f"\n[GAIE] Recall@{args.topk}: {gaie_metrics['Recall']:.4f}")
print(f"[GAIE] NDCG@{args.topk}:   {gaie_metrics['NDCG']:.4f}")

In [ ]:
gaie_mi = gaie_ft_coach.test_unlearn(gaie_ft_coach.model, prefix='[GAIE] Final Evaluation')

In [ ]:
all_results['GAIE'] = {
    'Recall': gaie_metrics['Recall'], 'NDCG': gaie_metrics['NDCG'],
    'MI_BF': gaie_mi['mi_bf'], 'MI_NG': gaie_mi['mi_ng'],
    'BeforeProb': gaie_mi['avg_before_prob'], 'AfterProb': gaie_mi['avg_after_prob'],
    'NegProb': gaie_mi['avg_neg_prob'],
    'UnlearnTime': gaie_unlearn_time, 'FinetuneTime': gaie_ft_time,
    'PeakMemGB': max(gaie_unlearn_mem, gaie_ft_mem),
}
print("GAIE results stored.")

del gaie_coach, gaie_ft_coach, handler_unlearn_gaie, handler_ft_gaie, handler_eval_gaie
t.cuda.empty_cache()
print("GAIE objects freed, GPU cache cleared.")

---
---
# Pipeline 4: HIE (Hypernetwork-Based Influence Encoder)

**Slowest pipeline** — ~40M params from HyperNetwork, 3 losses (BPR + unlearn + alignment).

Influence graph → GNN → mean-pool → latent z → HyperNetwork H(z) → W_u → ΔE = W_u ⊙ E

The HyperNetwork generates per-node weight matrices, making this significantly heavier.

## HIE — Step 2a: Unlearn

In [ ]:

# ============================================================
# Hyperparameters for HIE unlearning on Yelp2018
# ============================================================

args.trained_model = PRETRAINED_PATH

args.epoch = 30
args.batch = 4096
args.lr = 1e-3
args.pretrain_drop_rate = 0.1
args.test_drop_rate = 0.003
args.sim_epoch = 10
args.tst_epoch = 5
args.bpr_wei = 1.0
args.unlearn_wei = 0.3
args.align_wei = 0.08
args.align_temp = 10.0
args.align_type = 'v2'
args.unlearn_type = 'v1'
args.overall_withdraw_rate = 0.1
args.withdraw_rate_init = 1
args.hyper_temp = 1.0
args.unlearn_ssl = 1e-3
args.layer_mlp = 2
args.leaky = 0.99
args.act = 'leaky'
args.unlearn_layer = 0
args.perf_degrade = 0.4
args.fineTune = False
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_hie_unlearn'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_unlearn_hie = DataHandler()
handler_unlearn_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

print(f"Users: {args.user}, Items: {args.item}")
print(f"Training edges (after drop): {handler_unlearn_hie.trn_loader.dataset.__len__()}")
print(f"Dropped edges: {len(handler_unlearn_hie.dropped_edges[0])}")
print(f"Picked (retained) edges: {len(handler_unlearn_hie.picked_edges[0])}")

In [ ]:
from unlearning.hie_unlearn import Coach as HIECoach

hie_coach = HIECoach(handler_unlearn_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_coach.run()
hie_unlearn_time = time.time() - _t0
hie_unlearn_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE unlearn] {hie_unlearn_time:.1f}s, peak GPU mem {hie_unlearn_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Unlearning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## HIE — Step 2b: Fine-Tune

In [ ]:

args.model_2_finetune = './ckpt/yelp_hie_unlearn'

args.fineTune = True
args.epoch = 15
args.lr = 1e-3
args.batch = 4096
args.unlearn_wei = 0.15
args.align_wei = 0.08
args.align_temp = 10.0
args.perf_degrade = 0.5
args.sim_epoch = 10
args.tst_epoch = 5
args.adversarial_attack = True
args.adv_method = 'lightgcn'

args.save_path = './ckpt/yelp_hie_ft'

t.manual_seed(args.seed)
t.cuda.manual_seed_all(args.seed)
t.backends.cudnn.deterministic = True
np.random.seed(args.seed)
random.seed(args.seed)

print_args(args)


In [ ]:
handler_ft_hie = DataHandler()
handler_ft_hie.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)

hie_ft_coach = HIECoach(handler_ft_hie)
t.cuda.reset_peak_memory_stats()
_t0 = time.time()
hie_ft_coach.run()
hie_ft_time = time.time() - _t0
hie_ft_mem = t.cuda.max_memory_allocated() / 1e9
print(f"[HIE finetune] {hie_ft_time:.1f}s, peak GPU mem {hie_ft_mem:.2f} GB")

print("\n" + "="*60)
print("HIE Fine-tuning complete!")
print(f"Model saved to: {args.save_path}")
print("="*60)

## HIE — Step 3: Evaluate

In [ ]:
handler_eval_hie = DataHandler()
handler_eval_hie.load_data(drop_rate=args.test_drop_rate, adv_attack=True)

save_path = args.save_path
if not save_path.endswith('.mod'):
    save_path = save_path + '.mod'

if os.path.exists(save_path):
    ckp = t.load(save_path, weights_only=False)
    hie_ft_coach.model = ckp['model'].cuda()
    print(f"Loaded fine-tuned HIE model from {save_path}")
else:
    print("Using in-memory fine-tuned model")

hie_ft_coach.handler = handler_eval_hie

In [ ]:
hie_metrics = hie_ft_coach.tst_epoch(hie_ft_coach.model)
print(f"\n[HIE] Recall@{args.topk}: {hie_metrics['Recall']:.4f}")
print(f"[HIE] NDCG@{args.topk}:   {hie_metrics['NDCG']:.4f}")

In [ ]:
hie_mi = hie_ft_coach.test_unlearn(hie_ft_coach.model, prefix='[HIE] Final Evaluation')

In [ ]:
all_results['HIE'] = {
    'Recall': hie_metrics['Recall'], 'NDCG': hie_metrics['NDCG'],
    'MI_BF': hie_mi['mi_bf'], 'MI_NG': hie_mi['mi_ng'],
    'BeforeProb': hie_mi['avg_before_prob'], 'AfterProb': hie_mi['avg_after_prob'],
    'NegProb': hie_mi['avg_neg_prob'],
    'UnlearnTime': hie_unlearn_time, 'FinetuneTime': hie_ft_time,
    'PeakMemGB': max(hie_unlearn_mem, hie_ft_mem),
}
print("HIE results stored.")

del hie_coach, hie_ft_coach, handler_unlearn_hie, handler_ft_hie, handler_eval_hie
t.cuda.empty_cache()
print("HIE objects freed, GPU cache cleared.")

---
---
# Final Comparison: All 4 Pipelines

In [ ]:
import json

print("\n" + "#" * 110)
print("#" + " FINAL COMPARISON — attacked-backbone protocol (UnlearnRec Sec. 4.1.4) ".center(108) + "#")
print("#" * 110)
print(f"#  Dataset:  {args.data} ({args.user} users, {args.item} items)")
print(f"#  Backbone: LightGCN ({args.latdim}-dim, {args.gnn_layer} layers), trained on attacked graph (adv method: {args.adv_method})")
print(f"#  Unlearn target: all injected adversarial edges")
print("#" + "-" * 108 + "#")
header = (f"#  {'Method':<9} {'Recall@20':>10} {'NDCG@20':>9} {'MI-BF':>8} {'MI-NG':>8} "
          f"{'P(before)':>10} {'P(after)':>9} {'P(neg)':>8} {'Time(s)':>9} {'Mem(GB)':>8}  #")
print(header)
print("#" + "-" * 108 + "#")

for name in ['Retrain', 'AIE', 'CIE', 'GAIE', 'HIE']:
    r = all_results.get(name, {})
    def f(key, fmt='{:.4f}'):
        return fmt.format(r[key]) if key in r else 'N/A'
    total_time = (r.get('UnlearnTime', 0) or 0) + (r.get('FinetuneTime', 0) or 0)
    time_s = f"{total_time:.1f}" if 'UnlearnTime' in r else 'N/A'
    print(f"#  {name:<9} {f('Recall'):>10} {f('NDCG'):>9} {f('MI_BF'):>8} {f('MI_NG'):>8} "
          f"{f('BeforeProb'):>10} {f('AfterProb'):>9} {f('NegProb'):>8} {time_s:>9} {f('PeakMemGB', '{:.2f}'):>8}  #")

print("#" + "-" * 108 + "#")
print("#  Sanity check: P(before) should sit near positive-edge level (edges were trained on);         #")
print("#  a method truly forgets when P(after) <= P(neg). Speedup = Retrain time / method time.        #")
print("#" * 110)

os.makedirs('./logs', exist_ok=True)
out = {
    'dataset': args.data, 'backbone': 'lightgcn', 'latdim': args.latdim,
    'gnn_layer': args.gnn_layer, 'seed': args.seed, 'adv_method': args.adv_method,
    'protocol': 'attacked-backbone (UnlearnRec Sec 4.1.4)', 'results': all_results,
}
res_file = f'./logs/final_results_{args.data}.json'
with open(res_file, 'w') as fs:
    json.dump(out, fs, indent=2)
print(f"\nResults saved to {res_file} — download this from Kaggle output for the paper.")
